# 02 · Exploratory Data Analysis

**Input:** `../data/processed/master_cleaned_2015_2025.csv` (from `01_data_cleaning.ipynb`)  
**Output:** charts in `../outputs/eda_YYYY_MM_DD/figures/` + one `.md` per chart + `CHART_EXPLANATIONS.md` and `figures.json`.

This is just descriptive stats. It does not prove crop loss, forage or forecast skill. The ML target is a synthesized low-NDVI label.


In [21]:
import hashlib, json
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROC_DIR = Path("../data/processed")
RUN_DIR = Path("../outputs") / f"eda_{pd.Timestamp.today():%Y_%m_%d}"
FIG_DIR = RUN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("default")
plt.rcParams.update({"figure.dpi": 140, "font.size": 9, "axes.titlesize": 10,
                     "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

PANEL_PATH = PROC_DIR / "master_cleaned_2015_2025.csv"
if not PANEL_PATH.exists():
    raise FileNotFoundError(f"{PANEL_PATH} not found. Run 01_data_cleaning.ipynb first.")
PANEL = pd.read_csv(PANEL_PATH)

# Make date column from year/month
PANEL["date"] = pd.to_datetime(dict(year=PANEL["year"], month=PANEL["month"], day=1))

# Column names differ between runs (temp_mean_c vs temperature_mean, etc.). Pick what exists.
TEMP_COL = "temp_mean_c" if "temp_mean_c" in PANEL.columns else "temperature_mean"
RAIN_COL = "rainfall_mm" if "rainfall_mm" in PANEL.columns else "rainfall_sum"
SOIL_COL = "soil_moisture" if "soil_moisture" in PANEL.columns else "soil_moisture_mean"
NDVI_COL = "ndvi_mean"

EXPLANATIONS = {}

def save(fig, stem, title, explanation):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{stem}.png", bbox_inches="tight")
    plt.close(fig)
    EXPLANATIONS[stem] = f"# {title}\n\n![{title}](figures/{stem}.png)\n\n{explanation}\n"
    print(f"  {stem}.png")

print(f"Panel: {len(PANEL):,} rows | {PANEL['county'].nunique()} counties | {PANEL['date'].min():%Y-%m} to {PANEL['date'].max():%Y-%m}")
print(f"Output: {RUN_DIR}")
print(f"Columns: {list(PANEL.columns)}")
print(f"SHA256: {hashlib.sha256(PANEL_PATH.read_bytes()).hexdigest()[:16]}...")


Panel: 6,204 rows | 47 counties | 2015-01 to 2025-12
Output: ../outputs/eda_2026_11_27
Columns: ['county', 'year', 'month', 'ndvi_mean', 'temp_mean_c', 'rainfall_mm', 'soil_moisture', 'total', 'farming', 'crop_production', 'livestock_production', 'aquaculture', 'fishing', 'irrigation', 'ndvi_mean_imputed', 'temperature_mean_imputed', 'rainfall_sum_imputed', 'soil_moisture_mean_imputed', 'year_month', 'date']
SHA256: d0fa0148c4aec920...


## 1 · Coverage

Check how many months each county has. Should be the same for all counties if the panel is complete.


In [22]:
counts = PANEL.groupby("county").size().sort_values()
expected = int(counts.median())  # works for both 132 and 293-row panels
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(range(len(counts)), counts.values, color="#2b6842")
ax.axhline(expected, color="#b23b3b", ls="--", lw=1, label=f"Median {expected} months")
ax.set(xticks=[], xlabel="47 counties (sorted)", ylabel="Months present", title="Coverage: months per county")
ax.legend(frameon=False)
save(fig, "01_coverage_by_county", "Coverage: months per county",
     f"Each bar is one county; dashed line is the median ({expected} months). "
     f"All 47 bars reach it, so the panel is balanced: {len(PANEL):,} rows = 47 x {expected}. "
     f"If a bar was short, lags in the ML code would be wrong.")


  01_coverage_by_county.png


## 2 · Seasonality

Average rainfall and greenness by calendar month.


In [23]:
monthly = PANEL.groupby("month")[[RAIN_COL, NDVI_COL]].mean(numeric_only=True)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].bar(monthly.index, monthly[RAIN_COL], color="#5b8fb9")
axes[0].set(xlabel="Month", ylabel="Mean rainfall (mm)", title="Rainfall by month", xticks=range(1,13))
axes[1].plot(monthly.index, monthly[NDVI_COL], marker="o", color="#2b6842")
axes[1].set(xlabel="Month", ylabel="Mean NDVI", title="Vegetation by month", xticks=range(1,13))
save(fig, "02_seasonality_rainfall_ndvi", "Rainfall and NDVI by month",
     f"Rain peaks in long rains (Mar-May) and short rains (Oct-Dec). NDVI follows with a lag. "
     f"Peak rain month {int(monthly[RAIN_COL].idxmax())}, peak NDVI month {int(monthly[NDVI_COL].idxmax())}.")


  02_seasonality_rainfall_ndvi.png


## 3 · Annual trend

Yearly average greenness and rainfall.


In [24]:
annual = PANEL.groupby("year")[[NDVI_COL, RAIN_COL]].mean(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(annual.index, annual[NDVI_COL], marker="o", color="#2b6842", label="NDVI")
ax.set(xlabel="Year", ylabel="Mean NDVI", title="Annual NDVI 2002-2025")
ax2 = ax.twinx()
ax2.plot(annual.index, annual[RAIN_COL], marker="s", color="#5b8fb9", alpha=0.7, label="Rainfall")
ax2.set_ylabel("Mean rainfall (mm)")
ax.grid(False)
ax.legend(frameon=False, loc="lower left")
save(fig, "03_annual_ndvi_rainfall", "Annual NDVI and rainfall",
     f"NDVI range {annual[NDVI_COL].min():.3f} to {annual[NDVI_COL].max():.3f}. "
     f"National mean is flat; most change is at county level, not national.")


  03_annual_ndvi_rainfall.png


## 4 · County ranking

Which counties are greenest on average?


In [25]:
ranks = PANEL.groupby("county")[NDVI_COL].mean().sort_values()
fig, ax = plt.subplots(figsize=(9, 7))
colors = ["#b23b3b" if v < ranks.quantile(0.25) else "#2b6842" if v > ranks.quantile(0.75) else "#9a9a9a" for v in ranks.values]
ax.barh(range(len(ranks)), ranks.values, color=colors)
ax.set(yticks=range(len(ranks)), yticklabels=ranks.index, xlabel="Mean NDVI", title="Counties by mean NDVI")
ax.tick_params(axis="y", labelsize=7)
save(fig, "04_county_ndvi_ranking", "Counties ranked by NDVI",
     f"Spread {ranks.min():.3f} ({ranks.index[0]}) to {ranks.max():.3f} ({ranks.index[-1]}). "
     f"Arid north (Turkana, Marsabit) low, highlands high. Model uses county-relative threshold.")


  04_county_ndvi_ranking.png


## 5 · Distributions

Histograms for the main variables.


In [26]:
cols = [(NDVI_COL, "NDVI"), (RAIN_COL, "mm"), (SOIL_COL, "0-1")]
cols = [(c,l) for c,l in cols if c in PANEL.columns]
fig, axes = plt.subplots(1, len(cols), figsize=(11, 3.2))
if len(cols) == 1:
    axes = [axes]
for ax, (col, unit) in zip(axes, cols):
    ax.hist(PANEL[col].dropna(), bins=50, color="#2b6842", alpha=0.8)
    ax.set(xlabel=f"{col} ({unit})", ylabel="Count", title=col)
save(fig, "05_distributions", "Distributions",
     f"NDVI avg {PANEL[NDVI_COL].mean():.2f}. Rainfall is right-skewed (tail to {PANEL[RAIN_COL].max():.0f} mm).")


  05_distributions.png


## 6 · Correlation

How the variables move together.


In [27]:
corr_cols = [c for c in [NDVI_COL, RAIN_COL, TEMP_COL, SOIL_COL] if c in PANEL.columns]
corr = PANEL[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(5.6, 4.6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticks(range(len(corr_cols)))
ax.set_yticklabels(corr_cols)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(corr.iloc[i,j]) > 0.6 else "black")
fig.colorbar(im, ax=ax, shrink=0.8)
# Safe handling for missing soil (soil is NaN before 2002-09)
soil_r = f"{corr.loc[NDVI_COL, SOIL_COL]:.2f}" if SOIL_COL in corr.columns else "n/a"
rain_r = f"{corr.loc[NDVI_COL, RAIN_COL]:.2f}" if RAIN_COL in corr.columns else "n/a"
save(fig, "06_correlation_matrix", "Correlation matrix",
     f"NDVI vs soil r={soil_r}, vs rain r={rain_r}.")
print(corr.loc[NDVI_COL].to_string())


  06_correlation_matrix.png
ndvi_mean        1.000000
rainfall_mm      0.518933
temp_mean_c     -0.583852
soil_moisture    0.707959


## 7 · What was imputed

Count of filled missing values.


In [28]:
corr_cols = [c for c in [NDVI_COL, RAIN_COL, TEMP_COL, SOIL_COL] if c in PANEL.columns]
corr = PANEL[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(5.6, 4.6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)

ax.set_xticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticks(range(len(corr_cols)))
ax.set_yticklabels(corr_cols)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(corr.iloc[i, j]) > 0.6 else "black")

fig.colorbar(im, ax=ax, shrink=0.8)

# safe text even if soil is all-NaN / missing
soil_r = f"{corr.loc[NDVI_COL, SOIL_COL]:.2f}" if SOIL_COL in corr.columns and SOIL_COL in corr.index else "n/a"
rain_r = f"{corr.loc[NDVI_COL, RAIN_COL]:.2f}" if RAIN_COL in corr.columns else "n/a"

save(fig, "06_correlation_matrix", "Correlation matrix",
     f"NDVI vs soil r={soil_r}, vs rain r={rain_r}. Soil has many NaNs early, so soil correlation is weak.")

  06_correlation_matrix.png


## 8 · Write explanations


In [29]:
for stem, text in EXPLANATIONS.items():
    (RUN_DIR / f"{stem}.md").write_text(text, encoding="utf-8")
header = f"""# EDA - {len(PANEL):,} rows, {PANEL['county'].nunique()} counties, {PANEL['date'].min():%Y-%m} to {PANEL['date'].max():%Y-%m}
Panel: master_cleaned_2015_2025.csv - {len(PANEL):,} rows.
These are descriptive stats only. ML target is synthesized low-NDVI.
---
"""
(RUN_DIR / "CHART_EXPLANATIONS.md").write_text(header + "\n---\n\n".join(EXPLANATIONS.values()), encoding="utf-8")
(RUN_DIR / "figures.json").write_text(json.dumps(
    [{"file": f"figures/{s}.png", "explanation": f"{s}.md", "title": EXPLANATIONS[s].splitlines()[0].lstrip("# ")} for s in EXPLANATIONS], indent=2), encoding="utf-8")
print(f"Wrote {len(EXPLANATIONS)} charts to {RUN_DIR}")


Wrote 6 charts to ../outputs/eda_2026_11_27


In [30]:
summary = pd.DataFrame([{"chart": s, "explanation": f"{s}.md"} for s in EXPLANATIONS])
display(summary)
print(f"Combined: {RUN_DIR / 'CHART_EXPLANATIONS.md'}")


,chart,explanation
0,01_coverage_by_county,01_coverage_by_county.md
1,02_seasonality_rainfall_ndvi,02_seasonality_rainfall_ndvi.md
2,03_annual_ndvi_rainfall,03_annual_ndvi_rainfall.md
3,04_county_ndvi_ranking,04_county_ndvi_ranking.md
4,05_distributions,05_distributions.md
5,06_correlation_matrix,06_correlation_matrix.md


Combined: ../outputs/eda_2026_11_27/CHART_EXPLANATIONS.md
